# Unit 4 Challenge – Tier 3


## 1. Sourcing and Loading

### 1.1 Importing Libraries

In [ ]:
# Let's import the pandas, numpy libraries as pd, and np respectively.
import pandas as pd
import numpy as np

# Load the pyplot collection of functions from matplotlib, as plt
from matplotlib import pyplot as plt

### 1.2 Loading the data

In [ ]:
# First, make a variable called url_LondonHousePrices, and assign it the following link, enclosed in quotation-marks as a string:
# https://data.london.gov.uk/download/uk-house-price-index/70ac0766-8902-4eb5-aab5-01951aaed773/UK%20House%20price%20index.xls

url_LondonHousePrices = "https://data.london.gov.uk/download/uk-house-price-index/70ac0766-8902-4eb5-aab5-01951aaed773/UK%20House%20price%20index.xls"

# The dataset we're interested in contains the Average prices of the houses, and is actually on a particular sheet of the Excel file.
# As a result, we need to specify the sheet name in the read_excel() method.
# Put this data into a variable called properties.
properties = pd.read_excel(url_LondonHousePrices, sheet_name='Average price', index_col=None)

## 2. Cleaning, Transforming, and Visualizing

### 2.1 Exploring the data

In [ ]:
print(properties.shape)
properties.head()

### 2.2 Cleaning the data (Part 1)

In [ ]:
# Transpose the dataframe so boroughs are rows and dates are columns
properties_T = properties.T

# Promote row 0 (the date headers) to column names
properties_T.columns = properties_T.iloc[0]

# Drop row 0 now that it has become the column headers
properties_T = properties_T.drop(0)

# Reset the index so borough names become a regular column
properties_T = properties_T.reset_index()

# Rename 'Unnamed: 0' to 'London_Borough' and the NaT column to 'ID'
properties_T = properties_T.rename(
    columns={'Unnamed: 0': 'London_Borough', pd.NaT: 'ID'}
)

properties_T.head()

### 2.3 Cleaning the data (Part 2)

In [ ]:
# Verify the column names look sensible
print(properties_T.columns[:6].tolist())
print(properties_T.shape)

### 2.4 Transforming the data

In [ ]:
# Melt the DataFrame so each row is one (borough, month, price) observation — tidy data!
clean_properties = pd.melt(
    properties_T,
    id_vars=['London_Borough', 'ID']
)

# Rename the melted columns
clean_properties = clean_properties.rename(
    columns={0: 'Month', 'value': 'Average_price'}
)

clean_properties.head()

In [ ]:
# Make sure Average_price values are floating point numbers
clean_properties['Average_price'] = pd.to_numeric(
    clean_properties['Average_price'], errors='coerce'
)
print(clean_properties.dtypes)

### 2.5 Cleaning the data (Part 3)

In [ ]:
# Check how many entries are in the London_Borough column
print("Unique boroughs:", clean_properties['London_Borough'].nunique())
print(clean_properties['London_Borough'].unique())

In [ ]:
# Remove null values
NaNFreeDF2 = clean_properties.dropna()

# A list of non-boroughs
nonBoroughs = [
    'Inner London', 'Outer London',
    'NORTH EAST', 'NORTH WEST', 'YORKS & THE HUMBER',
    'EAST MIDLANDS', 'WEST MIDLANDS',
    'EAST OF ENGLAND', 'LONDON', 'SOUTH EAST',
    'SOUTH WEST', 'England'
]

NaNFreeDF2 = NaNFreeDF2[~NaNFreeDF2.London_Borough.isin(nonBoroughs)]

print("Boroughs after cleaning:", NaNFreeDF2['London_Borough'].nunique())
print(NaNFreeDF2.shape)

df = NaNFreeDF2
df.head()

### 2.6 Visualizing the data

In [ ]:
# Extract the year from the Month column using a lambda function
df['Year'] = df['Month'].apply(lambda x: x.year if hasattr(x, 'year') else int(str(x)[:4]))
df.head()

In [ ]:
# Subset on Camden and plot Average Price over time
camden = df[df['London_Borough'] == 'Camden']
camden_yearly = camden.groupby('Year')['Average_price'].mean()

plt.figure(figsize=(12, 5))
plt.plot(camden_yearly.index, camden_yearly.values, color='steelblue', linewidth=2)
plt.title("Camden – Average House Price by Year", fontsize=14)
plt.xlabel("Year")
plt.ylabel("Average Price (£)")
plt.tight_layout()
plt.show()

## 3. Modeling

In [ ]:
# Using groupby to calculate the mean for each year and each borough
dfg = df.groupby(by=['London_Borough', 'Year']).mean()
dfg = dfg.reset_index()
dfg.head()

In [ ]:
def create_price_ratio(d):
    y1998 = float(d['Average_price'][d['Year'] == 1998])
    y2018 = float(d['Average_price'][d['Year'] == 2018])
    ratio = [y2018 / y1998]
    return ratio

# Quick test with Camden
camden_ratio = create_price_ratio(dfg[dfg['London_Borough'] == 'Camden'])
print(f"Camden price ratio (2018 / 1998): {camden_ratio[0]:.2f}x")

In [ ]:
# Iterate through all boroughs and calculate the ratio
final = {}
for b in dfg['London_Borough'].unique():
    borough = dfg[dfg['London_Borough'] == b]
    final[b] = create_price_ratio(borough)

print(final)

In [ ]:
# Convert to a DataFrame, transpose, reset index and rename
df_ratios = pd.DataFrame(final)
df_ratios_T = df_ratios.T
df_ratios = df_ratios_T.reset_index()
df_ratios.rename(columns={'index': 'Borough', 0: '2018'}, inplace=True)
df_ratios.head()

In [ ]:
# Sort in descending order and select the top 15 boroughs
top15 = df_ratios.sort_values(by='2018', ascending=False).head(15)
print(top15)

In [ ]:
# Plot the boroughs that have seen the greatest changes in price
plt.figure(figsize=(14, 6))
top15[['Borough', '2018']].plot(kind='bar', color='steelblue', edgecolor='white')
plt.title("Top 15 London Boroughs by House Price Growth (1998–2018)", fontsize=14)
plt.xlabel("Borough")
plt.ylabel("Price Ratio (2018 price / 1998 price)")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 4. Conclusion

*What can you conclude? Type your conclusions below.*

In [ ]:
The London borough with the greatest average increase in house prices
over the two decades from 1998 to 2018 is: Hackney. 6.2x increase

House prices there grew by a factor of approximately {multiplier:.2f}x,
meaning a property worth £100,000 in 1998 would have been worth
roughly £620k by 2018.